# QF 627 Programming and Computational Finance
## Lesson 08 | Mini-Mock Assessment: Individual Practice Session

***

> Good afternoon, Team 👋

> The purpose of the current assessment is to serve as a simulation of the actual assessment day. Its main goal is simple—to help you prepare effectively and to shed light on your current level of knowledge and expertise.

> Through this `mock` `mini-assessment`, you’ll get a chance to see how questions might be structured and how you may want to approach writing your answers on the actual assessment day. Please note that this is not the real assessment, and it will not be graded.

> Importantly, this exercise will help me better understand where each of you stands, so I can support you more effectively during our final two weeks of learning.

> Don’t feel pressured—the actual assessment will give you three hours, while this mini mock assessment will take only 70 minutes.

***

> Be sure to submit your work before the deadline: `3:10pm, November 21, 2025`. It is an open-book exercise, and is also a timed task. To be fair to all students, a late submission will incur a point reduction.

> Please note `your last name` for `naming your submission` file (e.g., `Roh.ipynb`)

> If you find that you cannot answer a question, it would be wise to move on to another question that you can answer, and to finish that one first. `Make the best use of the time available`. If you cannot fully answer all the questions, then do as much as you can.

***

> Rather than feeling pressured by the assessment, I hope you will enjoy the opportunity presented by the hands-on exercise. You will notice that `answering each question will further consolidate your learning`.

***

> I wish you the best for your individual assessment 🤞

***

### For standardization of your answers…

> Please execute the lines of code below before you start work on your answers.

In [1]:
# Our standardized printing options

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl

np.set_printoptions(precision = 3)

pd.set_option("display.float_format", lambda x: "%.3f" % x)

plt.style.use("ggplot")

mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.alpha"] = 0.25

mpl.rcParams["axes.facecolor"] = "white"

mpl.rcParams["legend.fontsize"] = 14

List of Questions

    IMPORTANT NOTE: 

### <font color = purple> <center> One of the key aspects of the current assessment involves data wrangling. 

### <font color = purple> <center> Please ensure that you correctly wrangle your data to obtain valid answers.

### <font color = purple> <center> When using a function, please write the function within the current script.

##  <font color = blue> 👉 Question 1. </font> Please use the datasets `qs_A.csv`, `qs_B.csv`, `qs_C.csv`. The time period for analysis is from October 2006 to December 2012.

### The strategy that you'll be developing is as follows: you create two separate Simple Moving Averages (SMA) of a time series with differing lookback periods (here, 40 days and 100 days). If the short moving average exceeds the long moving average then you go long, if the long moving average exceeds the short moving average then you exit.

```python
    df["Signal"] = 0
    df.loc[df["SMA_40"] > df["SMA_100"], "Signal"] = 1
    df.loc[df["SMA_40"] < df["SMA_100"], "Signal"] = 0
```

### On the days that the signal is 1 and the short moving average crosses the long moving average (for the period greater than the shortest moving average window), you'll buy a 100 shares. The days on which the signal is 0, the final result will be 0 as a result of the operation 100 x signal.

### For rolling statistics, set `min_periods` at `1` and `center` argument at `False`.

### Use `Adj Close` price.

### Let’s suppose that you started from a `$100,000` capital base for each of the three securities.

### Disregarding commission, how much will you have in the end in your account for each of the securities as a result of the current momentum-based trading?

### Below are the lines of code that lead to an answer:

In [2]:
a = pd.read_csv('qs_A.csv')
b = pd.read_csv('qs_B.csv')
c = pd.read_csv('qs_C.csv')

In [3]:
a ==b

,Date,Open,High,Low,Close,Adj Close,Volume
0,True,False,False,False,False,False,False
1,True,False,False,False,False,False,False
2,True,False,False,False,False,False,False
3,True,False,False,False,False,False,False
4,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...
1567,True,False,False,False,False,False,False
1568,True,False,False,False,False,False,False
1569,True,False,False,False,False,False,False
1570,True,False,False,False,False,False,False


In [4]:
a = a[['Date', 'Adj Close']].set_index('Date')
b = b[['Date', 'Adj Close']].set_index('Date')
c = c[['Date', 'Adj Close']].set_index('Date')

In [5]:
a

,Adj Close
Date,
2006-10-02,2.269
2006-10-03,2.246
2006-10-04,2.285
2006-10-05,2.268
2006-10-06,2.250
...,...
2012-12-21,15.883
2012-12-24,15.909
2012-12-26,15.689


In [6]:
def get_sma(df, sma_list):
    """
    Take DF and given list of SMA to enrich df when stock price is in first column
    """
    for window in sma_list:
        df[f"sma_{window}"] =\
        (
            df[df.columns[0]]
            .rolling(window = window)
            .mean()
        )

    return df

sma_tup = (40, 100)

a= get_sma(a, sma_tup)
b= get_sma(b, sma_tup)
c= get_sma(c, sma_tup)


tickers_dict = {'a': a, 'b': b, 'c': c}

In [7]:


def get_momentum_strategy(df, sma_tuple):
    """
    Apply single stock momentum strategy pipeline to price DataFrame `df`. 
    Price is in first column.
    Adds columns:
      positions, trade, passive_returns, strategy_returns,
      cum_returns, cum_strategy_returns
    Parameters:
      df        : pd.DataFrame with price columns for ticker name as the header
      sma_tuple : tuple of (short_window, long_window) for SMAs
    Returns:
      pd.DataFrame with the above columns
    """
    def get_sma(df, sma_list):
        """
        Take DF and given list of SMA to enrich df when stock price is in first column
        """
        for window in sma_list:
            df[f"sma_{window}"] =\
            (
                df[df.columns[0]]
                .rolling(window = window,
                         min_periods = 1,
                         center=False)
                .mean()
            )

        return df
    sma_short, sma_long = sma_tuple
    if sma_short >= sma_long:
        raise ValueError("sma_short must be less than sma_long")
    df = get_sma(df, list(sma_tuple)).dropna()

    df['positions'] =\
    (
        np.where((df[f'sma_{sma_short}'] > df[f'sma_{sma_long}']), 1, 0)
    )
    df['trade'] = \
    (
        df['positions'].diff().fillna(0)
    )
    if df.at[df.index[0], 'positions'] != 0:
        df.at[df.index[0], 'trade'] = df.at[df.index[0], 'positions']
    df['passive_returns'] =\
    (
        np.log(df[df.columns[0]] 
            / df[df.columns[0]].shift(1))
    ).fillna(0)
    df['strategy_returns'] =\
    (
        df['passive_returns'] * df['positions'].shift(1).fillna(0)
    )
    df['cum_returns'] =\
    (
        df['passive_returns'].cumsum().apply(np.exp).fillna(1)
    )
    df['cum_strategy_returns'] =\
    (
        df['strategy_returns'].cumsum().apply(np.exp).fillna(1)
    )

    return df

# sma_tup = (42, 252)
# get_momentum_strategy(df[['GS']], sma_tup)

In [8]:
final_cum_ret_dict = {}
init_cap = 100_000

for ticker, df in tickers_dict.items():
    tickers_dict[ticker] = get_momentum_strategy(df, sma_tup)
    final_cum_ret_dict[ticker] =\
        tickers_dict[ticker]['cum_strategy_returns'][-1] * init_cap

/var/folders/h2/r7qn2m9n1zb6y_0q191gdqth0000gn/T/ipykernel_74437/3644122845.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  tickers_dict[ticker]['cum_strategy_returns'][-1] * init_cap
/var/folders/h2/r7qn2m9n1zb6y_0q191gdqth0000gn/T/ipykernel_74437/3644122845.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  tickers_dict[ticker]['cum_strategy_returns'][-1] * init_cap
/var/folders/h2/r7qn2m9n1zb6y_0q191gdqth0000gn/T/ipykernel_74437/3644122845.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFr

In [9]:
final_cum_ret_dict

{'a': np.float64(430254.36424476735),
 'b': np.float64(135964.9290800499),
 'c': np.float64(77358.99192887974)}

### <font color = red> Answer 1 </font>


    A  : _$______________ 
    
    B  : _$______________ 
    
    C  : _$______________ 


In [10]:
for ticker, final_cap in final_cum_ret_dict.items():
    print(f'{ticker.upper()}  : ${final_cap:,.2f}')

A  : $430,254.36
B  : $135,964.93
C  : $77,358.99


##  <font color = blue> 👉 Questions 2 & 3. </font>  

### Calculate and visualize the maximum drawdowns and the longest drawdown periods for `A`, `B`, and `C`.

### Below are the lines of code that lead to an answer:

In [11]:
def get_drawdowns(cum_returns):
    """
    Calculate the Maximum Drawdown (MDD) and the Longest Drawdown Duration (LDD)
    from a cumulative returns SERIES.

    Parameters:
      cum_returns : pd.SERIES of cumulative returns
    Returns:
      tuple (mdd, ldd)
    Usage: 
      get_drawdowns(df['cum_strategy_returns'])
    """
    def compute_longest_drawdown_period(dd, cum_returns):
      periods =\
      pd.to_datetime(pd.Series(np.append(daily_drawdown[daily_drawdown == 0].index, daily_drawdown.index[-1: ]))).diff()
      
      return periods.max()
    
    daily_drawdown = cum_returns / cum_returns.cummax() - 1
    mdd = daily_drawdown.min()
    ldd = compute_longest_drawdown_period(daily_drawdown, cum_returns) # days
    return mdd, ldd



In [12]:
mdd_dict = {}
ldd_dict = {}

for ticker, df in tickers_dict.items():
    mdd, ldd = get_drawdowns(df['cum_strategy_returns'])
    mdd_dict[ticker] = mdd
    ldd_dict[ticker] = ldd


In [13]:
mdd

np.float64(-0.374799049375932)

In [14]:
# """
#  TODO max drawdown and duration table
# """
# # take date column out and cumsum for periods
# dd_reset = df.reset_index()
# dd_reset['period'] = (dd_reset['daily_drawdown'] == 0).cumsum()
# # find nodes with 0 value i.e. highpoints
# dd_nonzero = dd_reset[dd_reset['daily_drawdown'] != 0]
# # aggregate by period
# period_stats = dd_nonzero.groupby('period').agg(
#     start_date = ('Date', 'min'),
#     end_date = ('Date', 'max'),
#     avg_dd = ('daily_drawdown', 'mean'),
#     max_dd = ('daily_drawdown', 'min'),
#     duration=('daily_drawdown', 'count')
# ).sort_values(by='avg_dd',ascending=True)

### <font color = red> Answer 2 (`visualization component`; use `lets-plot`) is presented in the cell below: </font>

In [15]:
a = tickers_dict['a']

In [16]:
for ticker, df in tickers_dict.items():
    df['gross_max'] = df['cum_strategy_returns'].cummax()
    tickers_dict[ticker] = df

In [17]:


from lets_plot import *
LetsPlot.setup_html()

for ticker, df in tickers_dict.items():
    df_melt = df.reset_index()
    # aapl_melt = aapl_melt.rename(columns={'index':'Date'}). # optional depending if index is called 'Date'

    p =\
    (
        ggplot(df_melt, aes(x='Date')) +
        geom_line(aes(y='cum_strategy_returns'), color='grey', size=0.7) +
        geom_line(aes(y='gross_max'), color='orange', size=0.7) +   # amend sma as required
        # geom_line(aes(y='sma_200'), color='blue', size=0.7,linetype=2) +
        # geom_point(aes(y='AAPL'), data=a_melt[a_melt['trade']>0.0], color='blue', size=3) +       # amend trade as required
        # geom_point(aes(y='AAPL'), data=a_melt[a_melt['trade']<-0.0], color='orange', size=3) +
        ggtitle(f"Ticker {ticker.upper()} cum and gross max ret") +
        ylab("returns") +
        xlab("Date") +
        ggsize(1200, 500) 
    )
    display(p)

### <font color = red> Answer 3 </font>
    
    As to A,
    
    The maximum drawdown is about ____________ percentage points.
    The longest drawdown period lasts for _____________ days.
    
    As to B,
    
    The maximum drawdown is about ____________ percentage points.
    The longest drawdown period lasts for _____________ days.
    
    As to C,
    
    The maximum drawdown is about ____________ percentage points.
    The longest drawdown period lasts for _____________ days.


In [18]:
mdd

np.float64(-0.374799049375932)

In [19]:
for ticker, df in tickers_dict.items():
    print(f"    As to {ticker.upper()},")
    print(f"    The maximum drawdown is about {mdd_dict[ticker]:.2%} percentage points.)")

    As to A,
    The maximum drawdown is about -49.88% percentage points.)
    As to B,
    The maximum drawdown is about -49.74% percentage points.)
    As to C,
    The maximum drawdown is about -37.48% percentage points.)


###  <font color = blue> 👉 Question 4. </font> Using the current momentum strategy, which of the securities shows the greatest Sharpe ratio?

### Below are the lines of code that lead to an answer:

In [20]:
def compute_sharpe_ratio(daily_returns):
    """
    get_sharpe_ratio() | Calculate the annualized Sharpe ratio from a series of daily returns.
    Annualized Sharpe ratio computed as:
                sqrt(252) * mean(daily_returns) / std(daily_returns)
    - Assumes 252 trading days per year for annualization. Adjust the
      multiplier for a different convention.
    - Input should be returns (not prices). Convert prices to returns before
      calling this function.
    Parameters:
        daily_returns : array-like (pd.Series or np.ndarray)
    Returns:
        float
    Usage:
        compute_sharpe_ratio(df['strategy_returns'])
    """
    return np.sqrt(252) * daily_returns.mean() / daily_returns.std()

# compute_sharpe_ratio(ibm['strategy_returns'])

In [21]:
sharpe_dict ={}
for ticker, df in tickers_dict.items():
    sharpe_dict[ticker] = compute_sharpe_ratio(df['strategy_returns'])
    print(compute_sharpe_ratio(df['strategy_returns']))



0.8376749939619653
0.23484928514384334
-0.2376732142052218


### <font color = red> Answer 4 </font>

    The answer is ____________________________ .

In [ ]:
lambda x: x.max() == 

dict_keys(['a', 'b', 'c'])

In [36]:
print(f"the answer is {max(sharpe_dict, key=sharpe_dict.get).upper()} with {sharpe_dict[max(sharpe_dict, key=sharpe_dict.get)]}")

the answer is A with 0.8376749939619653


###  <font color = blue> 👉 Question 5. </font> Report compound annual growth rate (CAGR) for `A`, `B`, and `C`.

### Below are the lines of code that lead to an answer:

In [24]:
def compute_CAGR(cumulative_returns):
    """
    get_cagr() | Compute Compound Annual Growth Rate (CAGR) from a series of cumulative returns.
        - The function drops missing values and uses the first and last available
      observations to compute CAGR as:
          (last_value / first_value) ** (365.0 / n_days) - 1
      where n_days is the integer number of days between the first and last index.
        - Uses 365-day convention for annualization. For trading-day conventions
        adjust the exponent accordingly.
        - If the series contains a single observation or zero-day span, the result
        may be ill-defined (division by zero or power of inf). Consider checking
        the index span before calling.
    Parameters:
        cumulative_returns : pandas.Series. Time-indexed series of cumulative returns (e.g. cumulative growth factors)
    Returns:
        float
    Usage:
        compute_CAGR(df['cum_strategy_returns'])
    """

    cumulative_returns = cumulative_returns.dropna()
    n_of_days = (cumulative_returns.index[-1] - cumulative_returns.index[0]
                ).days
    cagr =\
    (
        (        
        cumulative_returns.iloc[-1] 
        /
        cumulative_returns.iloc[0]
        ) ** (365.0 / n_of_days)
        - 1
    )
    return cagr

In [25]:
for ticker, df in tickers_dict.items():
    df.index = pd.to_datetime(df.index)

In [26]:
(tickers_dict['a'].index[-1] - tickers_dict['a'].index[0]).days

2279

In [27]:
cagr_dict ={}
for ticker, df in tickers_dict.items():
    cagr_dict[ticker] = compute_CAGR(df['cum_strategy_returns'])
    print(cagr_dict[ticker])


0.26326990863159483
0.050435476289216075
-0.04028094655128589


### <font color = red> Answer 5 </font>


    A  : _____________%__ 
    
    B : _____________%__ 
    
    C  : _____________%__ 


In [28]:
for ticker, cagr in cagr_dict.items():
    print(f"{ticker.upper()} :  {cagr:.2%}")

A :  26.33%
B :  5.04%
C :  -4.03%


    IMPORTANT NOTE: 

### <font color = purple> <center> Prior to submitting, ensure that you execute the following command to present your workspace.

### <font color = purple> <center> Before submission, ensure that your responses are entered into the designated cells provided for answering.

In [29]:
%whos

Variable                     Type         Data/Info
---------------------------------------------------
GGBunch                      type         <class 'lets_plot.plot.plot.GGBunch'>
LetsPlot                     type         <class 'lets_plot.LetsPlot'>
a                            DataFrame    Shape: (1572, 10)
aes                          function     <function aes at 0x10df160c0>
arrow                        function     <function arrow at 0x10e1222a0>
as_discrete                  function     <function as_discrete at 0x10e10b380>
b                            DataFrame    Shape: (1572, 3)
c                            DataFrame    Shape: (1572, 3)
cagr                         float64      Shape: ()
cagr_dict                    dict         n=3
compute_CAGR                 function     <function compute_CAGR at 0x10459a340>
compute_sharpe_ratio         function     <function compute_sharpe_ratio at 0x10ebb7f60>
coord_cartesian              function     <function coord_cartesian at 0x

### <font color = green> 💯 Thank you for putting your efforts into our mini-mock individual assessment questions 😊